# 06 — Model-Version Tagging and Mixed-Batch Flagging

Companion notebook to `07-production-resilience-and-operational-engineering.md`, bug narrative 4:
*"A retrained model silently deployed without updating its version tag."* Also implements the
model-version half of chapter 06 Part 4's `source_document_version` schema field -- the "which
version of the model produced this citation" axis, distinct from chapter 06's own "which version of
the source document" axis and from `05_content_hash_reprocessing_demo.ipynb`'s dedup concern.

This notebook demonstrates the bug **and** the fix, both runnably:

1. A **naive** deployment path that swaps a retrained CNN classifier onto the same endpoint
   name/alias without a version identifier -- reproducing the exact failure chapter 07 describes:
   every detection produced afterward is indistinguishable, in the data, from one produced by the old
   model, so an accuracy shift on a downstream audit can't be attributed to the model change versus
   anything else.
2. A **fixed** deployment path that requires a version identifier on every deploy (rejecting a push
   that doesn't carry one) and stamps every detection with the model version that produced it.
3. `flag_mixed_version_batches()` -- a check a review/audit dashboard would run on any sampled batch
   of detections, flagging a batch that spans more than one `cnn_model_version` as needing separate
   accuracy attribution rather than being scored as one undifferentiated population.

Fully offline: numpy and the standard library only -- no real Sagemaker endpoint, no real model
artifacts, no external calls.

In [1]:
import numpy as np

rng = np.random.default_rng(3)
print("Imports OK")

Imports OK


## Step 1 — Two deployment paths: naive (no enforced version) vs. fixed (version required)

`NaiveDeploymentRegistry` mirrors chapter 07's bug exactly: `deploy()` accepts an optional version
identifier and silently proceeds without one if it's omitted -- the same "endpoint name/alias stays
the same, nothing forces a version bump" gap the chapter describes. `FixedDeploymentRegistry` is the
proposed fix: a hard deployment-pipeline rule that **rejects** any deploy missing a version
identifier.

In [2]:
class NaiveDeploymentRegistry:
    """Reproduces chapter 07's bug: deploying a retrained model to the SAME endpoint name without
    a version identifier is silently accepted."""
    def __init__(self, endpoint_name: str):
        self.endpoint_name = endpoint_name
        self.current_version = None   # never required to be set

    def deploy(self, version_identifier=None):
        # No validation at all -- a retrain-and-redeploy with version_identifier=None just... works.
        self.current_version = version_identifier

    def current_model_version(self):
        return self.current_version   # frequently None, or stale from whenever it WAS last set


class FixedDeploymentRegistry:
    """Chapter 07's proposed fix: 'a hard deployment-pipeline rule that no model artifact reaches
    the serving endpoint without a version identifier bundled into the deployment (and
    rejected/blocked otherwise).'"""
    def __init__(self, endpoint_name: str):
        self.endpoint_name = endpoint_name
        self.current_version = None
        self.deploy_history = []

    def deploy(self, version_identifier: str = None):
        if not version_identifier:
            raise ValueError(
                f"Deploy to endpoint '{self.endpoint_name}' REJECTED: no version_identifier supplied. "
                f"Every model push must carry a version identifier -- this is a required deployment-"
                f"pipeline gate, not an optional convention."
            )
        self.current_version = version_identifier
        self.deploy_history.append(version_identifier)

    def current_model_version(self):
        return self.current_version


print("NaiveDeploymentRegistry, FixedDeploymentRegistry defined")

NaiveDeploymentRegistry, FixedDeploymentRegistry defined


In [3]:
naive_cnn_endpoint = NaiveDeploymentRegistry("superscript-cnn-prod-endpoint")
naive_cnn_endpoint.deploy(version_identifier="cnn-classifier-2024-06")
print("Naive registry after first deploy:", naive_cnn_endpoint.current_model_version())

# The retrain-and-redeploy from the bug narrative: same endpoint, no version identifier supplied.
naive_cnn_endpoint.deploy()
print("Naive registry after RETRAINED model redeployed with no version tag:", naive_cnn_endpoint.current_model_version())

fixed_cnn_endpoint = FixedDeploymentRegistry("superscript-cnn-prod-endpoint")
fixed_cnn_endpoint.deploy(version_identifier="cnn-classifier-2024-06")
print()
print("Fixed registry after first deploy:", fixed_cnn_endpoint.current_model_version())

try:
    fixed_cnn_endpoint.deploy()   # the identical mistake -- retrain, redeploy, forget the version tag
    raised = False
except ValueError as e:
    raised = True
    print("Fixed registry correctly REJECTED the untagged redeploy:")
    print(" ", e)

assert raised, "the fixed deployment pipeline must reject a deploy with no version identifier"
assert naive_cnn_endpoint.current_model_version() is None, "reproducing the bug: the naive registry silently loses version attribution"
print()
print("Confirmed: the identical operator mistake (retrain, redeploy, forget to tag) is silently "
      "accepted by the naive registry and outright rejected by the fixed one.")

Naive registry after first deploy: cnn-classifier-2024-06
Naive registry after RETRAINED model redeployed with no version tag: None

Fixed registry after first deploy: cnn-classifier-2024-06
Fixed registry correctly REJECTED the untagged redeploy:
  Deploy to endpoint 'superscript-cnn-prod-endpoint' REJECTED: no version_identifier supplied. Every model push must carry a version identifier -- this is a required deployment-pipeline gate, not an optional convention.

Confirmed: the identical operator mistake (retrain, redeploy, forget to tag) is silently accepted by the naive registry and outright rejected by the fixed one.


## Step 2 — A batch of citation detections spanning a real model swap

Simulate 60 days of detections. A genuinely **better** CNN classifier (higher true accuracy) is
retrained and redeployed at day 30. Each individual detection either does or doesn't get a
`cnn_model_version` tag depending on which registry produced it -- exactly the fork chapter 07
describes: the same underlying event (a model swap), tagged in one pipeline and untagged in the
other.

In [4]:
def simulate_detection_batch(registry, deploy_day: int, n_days: int = 60, detections_per_day: int = 20,
                               accuracy_before: float = 0.83, accuracy_after: float = 0.90):
    """Each 'detection' records whether the classifier's true/false call was correct (drawn from
    the model actually live that day) and whatever cnn_model_version the registry reports at the
    moment of detection. At the deploy boundary, the OPERATOR makes the identical mistake against
    both registries -- redeploys the retrained model without passing a version identifier. The
    naive registry silently accepts this (the bug). The fixed registry rejects it immediately,
    forcing the operator to correct the deploy with the real version before it can proceed --
    which is the fix actually taking effect, not just being available."""
    records = []
    new_version = "cnn-classifier-2024-09-retrained"
    for day in range(n_days):
        if day == deploy_day:
            try:
                registry.deploy()   # the operator's actual mistake: forgets to pass a version
            except ValueError:
                registry.deploy(version_identifier=new_version)   # caught immediately -- forced to correct it
        true_accuracy = accuracy_before if day < deploy_day else accuracy_after
        for _ in range(detections_per_day):
            correct = bool(rng.random() < true_accuracy)
            records.append({
                "day": day,
                "correct": correct,
                "cnn_model_version": registry.current_model_version(),
            })
    return records


naive_registry = NaiveDeploymentRegistry("superscript-cnn-prod-endpoint")
naive_registry.deploy(version_identifier="cnn-classifier-2024-06")
naive_records = simulate_detection_batch(naive_registry, deploy_day=30)

fixed_registry = FixedDeploymentRegistry("superscript-cnn-prod-endpoint")
fixed_registry.deploy(version_identifier="cnn-classifier-2024-06")
fixed_records = simulate_detection_batch(fixed_registry, deploy_day=30)

naive_versions_seen = sorted({r["cnn_model_version"] for r in naive_records}, key=lambda v: (v is None, v))
fixed_versions_seen = sorted({r["cnn_model_version"] for r in fixed_records}, key=lambda v: (v is None, v))
print("Distinct cnn_model_version values in the naive batch:", naive_versions_seen)
print("Distinct cnn_model_version values in the fixed batch:", fixed_versions_seen)

assert naive_versions_seen == ["cnn-classifier-2024-06", None], (
    "reproducing the bug: pre-swap detections keep the real v1 tag, but EVERY detection from the "
    "deploy boundary onward silently loses version attribution to None -- the retrained model's own "
    "detections carry no version at all, not even a stale/wrong one"
)
assert len(fixed_versions_seen) == 2, "the fixed-path batch should show exactly the two real model versions"
print()
print("Confirmed: in the naive batch, every detection from day 30 onward -- produced by the genuinely "
      "retrained, more accurate model -- is tagged None. There's no way to tell those detections apart "
      "from a data-quality problem, a logging outage, or anything else that produces a missing tag. "
      "The fixed batch correctly shows both real model versions throughout.")

Distinct cnn_model_version values in the naive batch: ['cnn-classifier-2024-06', None]
Distinct cnn_model_version values in the fixed batch: ['cnn-classifier-2024-06', 'cnn-classifier-2024-09-retrained']

Confirmed: in the naive batch, every detection from day 30 onward -- produced by the genuinely retrained, more accurate model -- is tagged None. There's no way to tell those detections apart from a data-quality problem, a logging outage, or anything else that produces a missing tag. The fixed batch correctly shows both real model versions throughout.


## Step 3 — Can a downstream accuracy audit attribute the shift to the model change?

Chapter 07's exact claim: *"If accuracy on a downstream sample audit then shifts (better or worse),
there is no way to attribute that shift to the model change versus something else... because nothing
in the recorded output says which model produced which detection."* Note precisely what that does and
doesn't mean: a reviewer grouping the naive batch by `cnn_model_version` will still notice something
changed at the boundary where the tag disappears -- but a bare `None` carries **no identifying
information** about *what* changed. It's exactly as consistent with "a retrained model shipped" as
with "the version-tagging code broke," "a new document source bypasses tagging," or "a logging
outage started dropping the field" -- the reviewer has a symptom, not an attributable cause. The
fixed batch, by contrast, names the actual model behind every detection.

In [5]:
import pandas as pd

naive_df = pd.DataFrame(naive_records)
fixed_df = pd.DataFrame(fixed_records)

overall_accuracy_before = naive_df[naive_df["day"] < 30]["correct"].mean()
overall_accuracy_after = naive_df[naive_df["day"] >= 30]["correct"].mean()
print(f"Naive batch: accuracy before day 30 = {overall_accuracy_before:.1%}, after = {overall_accuracy_after:.1%}")
print()
print("Grouping the naive batch by cnn_model_version:")
naive_by_version = naive_df.groupby("cnn_model_version", dropna=False)["correct"].agg(["mean", "count"])
print(naive_by_version)
print()
print("There IS a visible split -- but one side of it is bare None. A None tag is consistent with a "
      "retrained model, a tagging-code regression, a new document source, or a logging outage; the "
      "recorded data cannot distinguish any of those hypotheses from each other. Knowing 'something "
      "changed at this boundary' is not the same as knowing WHAT changed, and a None tag can never "
      "answer the second question.")

print()
print("=" * 70)
print()

fixed_by_version = fixed_df.groupby("cnn_model_version")["correct"].agg(["mean", "count"])
print("Fixed batch, grouped by cnn_model_version:")
print(fixed_by_version)

assert naive_df["cnn_model_version"].isna().sum() > 0, "the naive batch's post-swap detections should be untagged (None)"
assert len(fixed_by_version) == 2, "the fixed batch must cleanly separate the two model versions"
accuracy_gap = fixed_by_version.loc["cnn-classifier-2024-09-retrained", "mean"] - fixed_by_version.loc["cnn-classifier-2024-06", "mean"]
assert accuracy_gap > 0.03, "the retrained model's accuracy improvement should be clearly attributable in the fixed batch"
print()
print(f"Confirmed: the fixed batch directly and unambiguously attributes a {accuracy_gap:.1%} accuracy "
      "improvement to cnn-classifier-2024-09-retrained BY NAME -- the naive batch, at best, can only "
      "say 'something changed here,' never what.")

Naive batch: accuracy before day 30 = 85.2%, after = 88.8%

Grouping the naive batch by cnn_model_version:
                            mean  count
cnn_model_version                      
cnn-classifier-2024-06  0.851667    600
NaN                     0.888333    600

There IS a visible split -- but one side of it is bare None. A None tag is consistent with a retrained model, a tagging-code regression, a new document source, or a logging outage; the recorded data cannot distinguish any of those hypotheses from each other. Knowing 'something changed at this boundary' is not the same as knowing WHAT changed, and a None tag can never answer the second question.


Fixed batch, grouped by cnn_model_version:
                                      mean  count
cnn_model_version                                
cnn-classifier-2024-06            0.813333    600
cnn-classifier-2024-09-retrained  0.900000    600

Confirmed: the fixed batch directly and unambiguously attributes a 8.7% accuracy improve

## Step 4 — `flag_mixed_version_batches()`: a review-dashboard-facing check

A concrete, reusable piece a sample-audit or reporting dashboard would run on any batch of detections
pulled for review: if the batch spans more than one `cnn_model_version` (or `yolo_model_version`),
flag it -- so whoever is reading an accuracy number for that batch knows to split it by version
before drawing a conclusion, rather than silently averaging two different models' performance
together into one number.

In [6]:
def flag_mixed_version_batches(df: pd.DataFrame, version_column: str = "cnn_model_version",
                                 group_column: str = "day", window_size: int = 7) -> pd.DataFrame:
    """Groups detections into review windows (here, blocks of `window_size` consecutive days,
    standing in for whatever sampling window a real audit/dashboard batch would use) and flags any
    window whose detections span more than one distinct model version."""
    df = df.copy()
    df["window"] = df[group_column] // window_size
    rows = []
    for window, group in df.groupby("window"):
        distinct_versions = sorted(group[version_column].dropna().unique().tolist())
        rows.append({
            "window": int(window),
            "days": f"{group[group_column].min()}-{group[group_column].max()}",
            "n_detections": len(group),
            "distinct_versions": distinct_versions,
            "mixed_version_batch": len(distinct_versions) > 1,
        })
    return pd.DataFrame(rows)


fixed_window_report = flag_mixed_version_batches(fixed_df)
print(fixed_window_report.to_string(index=False))

mixed_windows = fixed_window_report[fixed_window_report["mixed_version_batch"]]
print()
print(f"Flagged {len(mixed_windows)} mixed-version window(s) out of {len(fixed_window_report)} total.")

assert len(mixed_windows) == 1, "exactly the window containing day 30 (the deploy boundary) should be flagged mixed-version"
assert mixed_windows.iloc[0]["days"] == "28-34"
print("Confirmed: only the single 7-day window straddling the actual deploy boundary (day 30) is "
      "flagged -- every other window correctly reports a single, unambiguous model version.")

 window  days  n_detections                                          distinct_versions  mixed_version_batch
      0   0-6           140                                   [cnn-classifier-2024-06]                False
      1  7-13           140                                   [cnn-classifier-2024-06]                False
      2 14-20           140                                   [cnn-classifier-2024-06]                False
      3 21-27           140                                   [cnn-classifier-2024-06]                False
      4 28-34           140 [cnn-classifier-2024-06, cnn-classifier-2024-09-retrained]                 True
      5 35-41           140                         [cnn-classifier-2024-09-retrained]                False
      6 42-48           140                         [cnn-classifier-2024-09-retrained]                False
      7 49-55           140                         [cnn-classifier-2024-09-retrained]                False
      8 56-59            80 

## Step 5 — Making the schema field required, not optional

Chapter 07's structural fix, alongside the deployment-pipeline gate: *"making `source_document_version`'s
model-version fields a required, non-nullable part of the structured citation output schema from day
one -- so it's structurally impossible to produce a citation record that can't be traced back to the
model that made it."* A minimal, runnable version of that constraint.

In [7]:
def make_citation_record(job_id: str, claim_text: str, marker: str,
                          yolo_model_version: str, cnn_model_version: str) -> dict:
    """cnn_model_version and yolo_model_version are REQUIRED positional-ish arguments with no
    default -- there is no way to call this constructor and produce a record missing either one,
    unlike the naive registry above, where a version was always merely optional."""
    if not yolo_model_version or not cnn_model_version:
        raise ValueError(
            "A citation record cannot be created without both yolo_model_version and "
            "cnn_model_version -- these fields are required, non-nullable parts of the schema."
        )
    return {
        "job_id": job_id, "claim_text": claim_text, "marker": marker,
        "source_document_version": {
            "yolo_model_version": yolo_model_version,
            "cnn_model_version": cnn_model_version,
        },
    }


good_record = make_citation_record(
    "job-901", "reduced onset by 40%", "1",
    yolo_model_version="yolo-v5-superscript-2024-03", cnn_model_version="cnn-classifier-2024-09-retrained",
)
print("Valid record:", good_record)

try:
    make_citation_record("job-902", "improved response rate by 25%", "2",
                          yolo_model_version="yolo-v5-superscript-2024-03", cnn_model_version=None)
    raised = False
except ValueError as e:
    raised = True
    print()
    print("Rejected an attempt to create an untraceable record:")
    print(" ", e)

assert raised, "constructing a citation record without a cnn_model_version must be impossible, not just discouraged"
print()
print("Confirmed: it is structurally impossible to construct a citation record that can't be traced "
      "back to the model that produced it -- the fix lives in the schema constructor itself, not in "
      "hoping every call site remembers to pass a version.")

Valid record: {'job_id': 'job-901', 'claim_text': 'reduced onset by 40%', 'marker': '1', 'source_document_version': {'yolo_model_version': 'yolo-v5-superscript-2024-03', 'cnn_model_version': 'cnn-classifier-2024-09-retrained'}}

Rejected an attempt to create an untraceable record:
  A citation record cannot be created without both yolo_model_version and cnn_model_version -- these fields are required, non-nullable parts of the schema.

Confirmed: it is structurally impossible to construct a citation record that can't be traced back to the model that produced it -- the fix lives in the schema constructor itself, not in hoping every call site remembers to pass a version.


## Tying it back

- Steps 1-2 reproduce chapter 07's bug narrative 4 exactly: the same operator mistake (retrain,
  redeploy to the same endpoint, forget the version tag) is silently accepted by a naive registry and
  outright rejected by one with a required-version deployment gate.
- Step 3 makes the chapter's attribution claim checkable: the naive batch's accuracy shift is real but
  genuinely unattributable to the model change from inside the data itself; the fixed batch attributes
  it cleanly by grouping on `cnn_model_version`.
- Step 4's `flag_mixed_version_batches()` is the piece a review dashboard would actually run --
  turning "does this batch span a model change" from a question a human has to remember to ask into
  an automatic check over any sampled review window.
- Step 5 closes the loop with chapter 06 Part 4's schema itself: making the version fields required,
  not optional, so the fix doesn't depend on every future call site remembering to pass one --
  structurally impossible beats "please remember," the same lesson this course's chapter 06 draws for
  the document-version half of this same underlying problem.